In [1]:
!pip install fairlearn pgmpy optuna -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.1/165.1 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 52.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.4.6 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.


In [2]:
import os, gc, warnings
import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score

warnings.filterwarnings("ignore")

SEED           = 42
DEVICE         = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TARGET_EO      = 0.05
MIN_ACC_FLOOR  = 0.55
N_ADV_SAMPLES  = 10
N_ENC_SAMPLES  = 16
N_TEST_SAMPLES = 40

PARAMS = {
    "german":      {"hidden_dim": 128,  "dropout": 0.20, "lr": 2e-4, "batch": 32,
                    "epochs": 3000, "lam_max": 18.0, "warmup": 150, "eval_every": 5,
                    "patience": 600},
    "compas":      {"hidden_dim": 256,  "dropout": 0.15, "lr": 5e-4, "batch": 128,
                    "epochs": 2000, "lam_max": 20.0, "warmup": 100, "eval_every": 5,
                    "patience": 500},
    "adult":       {"hidden_dim": 256,  "dropout": 0.15, "lr": 6e-4, "batch": 512,
                    "epochs": 1000, "lam_max": 14.0, "warmup": 50,  "eval_every": 5,
                    "patience": 250},
    "communities": {"hidden_dim": 512,  "dropout": 0.20, "lr": 2e-4, "batch": 64,
                    "epochs": 3000, "lam_max": 22.0, "warmup": 150, "eval_every": 5,
                    "patience": 600},
    "heart":       {"hidden_dim": 256,  "dropout": 0.20, "lr": 2e-4, "batch": 32,
                    "epochs": 3000, "lam_max": 20.0, "warmup": 150, "eval_every": 5,
                    "patience": 600},
}

DATASET_CONFIG = {
    "adult":       {"sens_attrs": [("sens_sex_train",  "sens_sex_test",  "sex"),
                                   ("sens_race_train", "sens_race_test", "race")],
                    "primary_sens": "sex"},
    "communities": {"sens_attrs": [("sens_race_train", "sens_race_test", "race")],
                    "primary_sens": "race"},
    "heart":       {"sens_attrs": [("sens_sex_train",  "sens_sex_test",  "sex"),
                                   ("sens_age_train",  "sens_age_test",  "age")],
                    "primary_sens": "sex"},
    "german":      {"sens_attrs": [("sens_age_train",  "sens_age_test",  "age"),
                                   ("sens_sex_train",  "sens_sex_test",  "sex")],
                    "primary_sens": "age"},
    "compas":      {"sens_attrs": [("sens_race_train", "sens_race_test", "race"),
                                   ("sens_sex_train",  "sens_sex_test",  "sex")],
                    "primary_sens": "race"},
}


def set_seed(s=SEED):
    torch.manual_seed(s)
    np.random.seed(s)

def to_dense(X):
    return X.toarray() if hasattr(X, "toarray") else np.array(X)

def ensure_binary(sv):
    sv = np.array(sv).flatten()
    u  = np.unique(sv)
    if len(u) == 1:
        return np.zeros(len(sv), dtype=int)
    if len(u) == 2:
        return (sv == u[1]).astype(int)
    maj = u[np.argmax([(sv == v).sum() for v in u])]
    return (sv != maj).astype(int)

def clean_numeric(s):
    s = pd.to_numeric(s, errors="coerce")
    return s.fillna(s.median() if s.notna().any() else 0)

def stratified_split(X, y, sens_list, test_size=0.2):
    key = np.array(y).astype(str).copy()
    for s in sens_list:
        key = key + "_" + np.array(s).astype(str)
    u, c = np.unique(key, return_counts=True)
    small = u[c < 2]
    key[np.isin(key, small)] = np.array(y)[np.isin(key, small)].astype(str)
    try:
        return train_test_split(np.arange(len(y)), test_size=test_size,
                                stratify=key, random_state=SEED)
    except Exception:
        return train_test_split(np.arange(len(y)), test_size=test_size,
                                stratify=y, random_state=SEED)

def build_val_split(y_train, frac=0.15):
    rng     = np.random.RandomState(SEED)
    val_idx = rng.choice(len(y_train), size=max(1, int(len(y_train) * frac)), replace=False)
    mask    = np.ones(len(y_train), dtype=bool)
    mask[val_idx] = False
    return np.where(mask)[0], val_idx

def compute_eo(y_true, y_pred, sv):
    sv = ensure_binary(sv)
    if len(np.unique(sv)) != 2:
        return 0.0
    tprs, fprs = [], []
    for g in [0, 1]:
        pos = (sv == g) & (y_true == 1)
        neg = (sv == g) & (y_true == 0)
        tprs.append(float(y_pred[pos].mean()) if pos.sum() > 0 else 0.0)
        fprs.append(float(y_pred[neg].mean()) if neg.sum() > 0 else 0.0)
    return max(abs(tprs[0] - tprs[1]), abs(fprs[0] - fprs[1]))


class Encoder(nn.Module):
    def __init__(self, in_dim, hidden_dim, dropout):
        super().__init__()
        h = hidden_dim
        self.net = nn.Sequential(
            nn.Linear(in_dim, h),      nn.BatchNorm1d(h),
            nn.LeakyReLU(0.1),         nn.Dropout(dropout),
            nn.Linear(h, h // 2),      nn.BatchNorm1d(h // 2),
            nn.LeakyReLU(0.1),         nn.Dropout(dropout * 0.5),
            nn.Linear(h // 2, h // 2), nn.LeakyReLU(0.1),
        )
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="leaky_relu")
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x)


class TaskHead(nn.Module):
    def __init__(self, rep_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(rep_dim, rep_dim // 2), nn.LeakyReLU(0.1),
            nn.Linear(rep_dim // 2, 1),
        )
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="leaky_relu")
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, rep):
        return self.net(rep).squeeze(1)


class Adversary(nn.Module):
    def __init__(self, rep_dim, hidden_dim, dropout):
        super().__init__()
        in_dim = rep_dim + 1
        h      = max(64, hidden_dim // 2)
        self.net = nn.Sequential(
            nn.Linear(in_dim, h),  nn.LeakyReLU(0.1), nn.Dropout(dropout),
            nn.Linear(h, h // 2), nn.LeakyReLU(0.1), nn.Dropout(dropout * 0.5),
            nn.Linear(h // 2, 1),
        )
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="leaky_relu")
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, rep, y):
        if y.dim() == 1: y = y.unsqueeze(1)
        return self.net(torch.cat([rep, y], dim=1)).squeeze(1)


class BayesianAdversary(nn.Module):
    def __init__(self, rep_dim, hidden_dim, dropout, prior_std=0.1):
        super().__init__()
        in_dim         = rep_dim + 1
        h              = max(64, hidden_dim // 2)
        self.prior_std = prior_std

        self.w1_mu  = nn.Parameter(torch.empty(h, in_dim))
        self.w1_rho = nn.Parameter(torch.full((h, in_dim), -3.0))
        self.b1_mu  = nn.Parameter(torch.zeros(h))
        self.b1_rho = nn.Parameter(torch.full((h,), -3.0))

        self.w2_mu  = nn.Parameter(torch.empty(h // 2, h))
        self.w2_rho = nn.Parameter(torch.full((h // 2, h), -3.0))
        self.b2_mu  = nn.Parameter(torch.zeros(h // 2))
        self.b2_rho = nn.Parameter(torch.full((h // 2,), -3.0))

        self.w3_mu  = nn.Parameter(torch.empty(1, h // 2))
        self.w3_rho = nn.Parameter(torch.full((1, h // 2), -3.0))
        self.b3_mu  = nn.Parameter(torch.zeros(1))
        self.b3_rho = nn.Parameter(torch.full((1,), -3.0))

        self.drop = nn.Dropout(dropout * 0.5)

        nn.init.kaiming_normal_(self.w1_mu, nonlinearity="leaky_relu")
        nn.init.kaiming_normal_(self.w2_mu, nonlinearity="leaky_relu")
        nn.init.kaiming_normal_(self.w3_mu, nonlinearity="leaky_relu")

    @staticmethod
    def _sigma(rho):
        return F.softplus(rho) + 1e-6

    def _sample_weight(self, mu, rho):
        return mu + self._sigma(rho) * torch.randn_like(mu)

    def forward_sampled(self, rep, y):
        if y.dim() == 1: y = y.unsqueeze(1)
        x  = torch.cat([rep, y], dim=1)
        w1 = self._sample_weight(self.w1_mu, self.w1_rho)
        b1 = self._sample_weight(self.b1_mu, self.b1_rho)
        w2 = self._sample_weight(self.w2_mu, self.w2_rho)
        b2 = self._sample_weight(self.b2_mu, self.b2_rho)
        w3 = self._sample_weight(self.w3_mu, self.w3_rho)
        b3 = self._sample_weight(self.b3_mu, self.b3_rho)
        h1 = self.drop(F.leaky_relu(F.linear(x,  w1, b1), 0.1))
        h2 = self.drop(F.leaky_relu(F.linear(h1, w2, b2), 0.1))
        return F.linear(h2, w3, b3).squeeze(1)

    def forward_mean(self, rep, y):
        if y.dim() == 1: y = y.unsqueeze(1)
        x  = torch.cat([rep, y], dim=1)
        h1 = F.leaky_relu(F.linear(x,  self.w1_mu, self.b1_mu), 0.1)
        h2 = F.leaky_relu(F.linear(h1, self.w2_mu, self.b2_mu), 0.1)
        return F.linear(h2, self.w3_mu, self.b3_mu).squeeze(1)

    def kl_divergence(self):
        kl = 0.0
        for mu, rho in [
            (self.w1_mu, self.w1_rho), (self.b1_mu, self.b1_rho),
            (self.w2_mu, self.w2_rho), (self.b2_mu, self.b2_rho),
            (self.w3_mu, self.w3_rho), (self.b3_mu, self.b3_rho),
        ]:
            sigma = self._sigma(rho)
            kl   += (torch.log(torch.tensor(self.prior_std, device=mu.device) / sigma)
                     + (sigma**2 + mu**2) / (2 * self.prior_std**2) - 0.5).sum()
        return kl

    def ensemble_logits(self, rep, y, n_samples):
        return torch.stack([self.forward_sampled(rep, y) for _ in range(n_samples)], dim=0)

    def forward(self, rep, y):
        return self.forward_sampled(rep, y)


def adversarial_training(
    X_tr, y_tr, primary_sv_tr,
    X_val, y_val, primary_sv_val,
    X_test, y_test, primary_sv_test,
    primary_name, params, baseline_eo,
    method="bayesian",
):
    tqdm.write(f"\n{'='*70}")
    tqdm.write(f"  [{method.upper()}] Adv Training  |  sens={primary_name}  |  baseline_EO={baseline_eo:.4f}")

    p = params
    set_seed()

    Xt  = torch.tensor(X_tr,           dtype=torch.float32).to(DEVICE)
    yt  = torch.tensor(y_tr,           dtype=torch.float32).to(DEVICE)
    st  = torch.tensor(primary_sv_tr,  dtype=torch.float32).to(DEVICE)
    Xv  = torch.tensor(X_val,          dtype=torch.float32).to(DEVICE)
    Xte = torch.tensor(X_test,         dtype=torch.float32).to(DEVICE)

    n_pos      = primary_sv_tr.sum()
    n_neg      = len(primary_sv_tr) - n_pos
    pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32).to(DEVICE)

    bce_task = nn.BCEWithLogitsLoss()
    bce_adv  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    loader = DataLoader(
        TensorDataset(Xt, yt, st),
        batch_size=p["batch"], shuffle=True, drop_last=True, num_workers=0,
    )
    n_batches = len(loader)

    rep_dim   = p["hidden_dim"] // 2
    encoder   = Encoder(X_tr.shape[1], p["hidden_dim"], p["dropout"]).to(DEVICE)
    task_head = TaskHead(rep_dim).to(DEVICE)

    if method == "bayesian":
        adversary = BayesianAdversary(rep_dim, p["hidden_dim"], p["dropout"]).to(DEVICE)
    else:
        adversary = Adversary(rep_dim, p["hidden_dim"], p["dropout"]).to(DEVICE)

    opt_enc = optim.AdamW(
        list(encoder.parameters()) + list(task_head.parameters()),
        lr=p["lr"], weight_decay=1e-4,
    )
    opt_adv = optim.AdamW(
        adversary.parameters(),
        lr=p["lr"] * 3.0, weight_decay=1e-5,
    )
    sched_enc = optim.lr_scheduler.CosineAnnealingLR(
        opt_enc, p["epochs"], eta_min=p["lr"] * 0.01,
    )
    sched_adv = optim.lr_scheduler.CosineAnnealingLR(
        opt_adv, p["epochs"], eta_min=p["lr"] * 0.1,
    )

    PRE_EPOCHS = 30 if baseline_eo > 0.25 else (5 if len(y_tr) > 5000 else 20)
    tqdm.write(f"  Pre-training adversary ({PRE_EPOCHS} epochs)...")
    for _ in range(PRE_EPOCHS):
        adversary.train()
        encoder.eval()
        for xb, yb, sb in loader:
            opt_adv.zero_grad(set_to_none=True)
            with torch.no_grad():
                rep = encoder(xb)
            if method == "bayesian":
                logits = adversary.forward_sampled(rep, yb)
                kl     = adversary.kl_divergence()
                loss   = bce_adv(logits, sb) + kl / n_batches
            else:
                loss = bce_adv(adversary(rep, yb), sb)
            loss.backward()
            nn.utils.clip_grad_norm_(adversary.parameters(), 1.0)
            opt_adv.step()

    best_eo        = np.inf
    best_state_enc = None
    best_state_th  = None
    best_composite = np.inf
    no_improve     = 0
    ema_val_eo     = None
    EMA_ALPHA      = 0.25

    pbar = tqdm(range(p["epochs"]), desc=f"  [{method[:3]}]", dynamic_ncols=True)

    for epoch in pbar:
        lam             = p["lam_max"] * min(1.0, epoch / max(p["warmup"], 1))
        bayesian_active = (method == "bayesian") and (epoch >= p["warmup"] // 2)

        # ── Phase 1: full adversary pass on frozen encoder reps ──────────────
        encoder.eval()
        task_head.eval()
        adversary.train()
        with torch.no_grad():
            all_reps = []
            all_ybs  = []
            all_sbs  = []
            for xb, yb, sb in loader:
                all_reps.append(encoder(xb))
                all_ybs.append(yb)
                all_sbs.append(sb)

        for rep_b, yb, sb in zip(all_reps, all_ybs, all_sbs):
            opt_adv.zero_grad(set_to_none=True)
            if method == "bayesian":
                logits   = adversary.forward_sampled(rep_b, yb)
                kl       = adversary.kl_divergence()
                adv_loss = bce_adv(logits, sb) + kl / n_batches
            else:
                adv_loss = bce_adv(adversary(rep_b, yb), sb)
            adv_loss.backward()
            nn.utils.clip_grad_norm_(adversary.parameters(), 1.0)
            opt_adv.step()

        # ── Phase 2: full encoder pass with frozen adversary ─────────────────
        encoder.train()
        task_head.train()
        adversary.eval()

        for xb, yb, sb in loader:
            opt_enc.zero_grad(set_to_none=True)
            rep       = encoder(xb)
            task_loss = bce_task(task_head(rep), yb)

            if bayesian_active:
                ensemble  = adversary.ensemble_logits(rep, yb, n_samples=N_ENC_SAMPLES)
                penalties = torch.stack(
                    [bce_adv(ensemble[k], sb) for k in range(N_ENC_SAMPLES)]
                )
                adv_mean           = penalties.mean()
                adv_var            = penalties.var().detach()
                uncertainty_weight = 1.0 / (1.0 + adv_var.clamp(min=1e-6))
                enc_loss           = task_loss - lam * uncertainty_weight * adv_mean
                clip_val           = 0.2
            else:
                if method == "bayesian":
                    logits_enc = adversary.forward_sampled(rep, yb)
                else:
                    logits_enc = adversary(rep, yb)
                enc_loss = task_loss - lam * bce_adv(logits_enc, sb)
                clip_val = 0.5

            enc_loss.backward()
            nn.utils.clip_grad_norm_(
                list(encoder.parameters()) + list(task_head.parameters()), clip_val,
            )
            opt_enc.step()

        sched_enc.step()
        sched_adv.step()

        if epoch % p["eval_every"] == 0:
            encoder.eval()
            task_head.eval()
            with torch.no_grad():
                if method == "bayesian":
                    pv = torch.stack([
                        torch.sigmoid(task_head(encoder(Xv)))
                        for _ in range(N_ADV_SAMPLES)
                    ]).mean(0).cpu().numpy()
                else:
                    pv = torch.sigmoid(task_head(encoder(Xv))).cpu().numpy()

            val_pred = (pv > 0.5).astype(int)
            val_eo   = compute_eo(y_val, val_pred, primary_sv_val)
            val_acc  = accuracy_score(y_val, val_pred)

            ema_val_eo  = val_eo if ema_val_eo is None else EMA_ALPHA * val_eo + (1 - EMA_ALPHA) * ema_val_eo
            acc_penalty = max(0.0, MIN_ACC_FLOOR + 0.05 - val_acc) * 5.0
            composite   = ema_val_eo + acc_penalty

            if composite < best_composite:
                best_composite = composite
                best_eo        = val_eo
                best_state_enc = {k: v.clone() for k, v in encoder.state_dict().items()}
                best_state_th  = {k: v.clone() for k, v in task_head.state_dict().items()}
                no_improve     = 0
            else:
                no_improve += p["eval_every"]

            pbar.set_postfix({
                "lam":  f"{lam:.2f}",
                "v_EO": f"{val_eo:.4f}",
                "ema":  f"{ema_val_eo:.4f}",
                "vacc": f"{val_acc:.4f}",
                "best": f"{best_eo:.4f}",
            })

            if best_eo <= TARGET_EO and no_improve >= p["patience"]:
                tqdm.write(f"\n  Early stop ep={epoch}")
                break

    if best_state_enc is not None:
        encoder.load_state_dict(best_state_enc)
        task_head.load_state_dict(best_state_th)

    encoder.eval()
    task_head.eval()
    with torch.no_grad():
        if method == "bayesian":
            proba_test = torch.stack([
                torch.sigmoid(task_head(encoder(Xte)))
                for _ in range(N_TEST_SAMPLES)
            ]).mean(0).cpu().numpy()
        else:
            proba_test = torch.sigmoid(task_head(encoder(Xte))).cpu().numpy()

    test_pred = (proba_test > 0.5).astype(int)
    eo_final  = compute_eo(y_test, test_pred, primary_sv_test)
    acc_final = accuracy_score(y_test, test_pred)

    status = "PASS" if eo_final < TARGET_EO else "FAIL"
    tqdm.write(
        f"  Baseline→Final: EO {baseline_eo:.4f} → {eo_final:.4f}  "
        f"acc={acc_final:.4f}  {status}"
    )

    return encoder, task_head, eo_final, acc_final


def run_pipeline(data, ds_key, method="bayesian"):
    set_seed()
    cfg          = DATASET_CONFIG[ds_key]
    p            = PARAMS[ds_key]
    primary_name = cfg["primary_sens"]

    X_train = to_dense(data["X_train_nn"])
    X_test  = to_dense(data["X_test_nn"])
    y_train = np.array(data["y_train"])
    y_test  = np.array(data["y_test"])

    sens_names         = [e[2] for e in cfg["sens_attrs"]]
    primary_idx        = sens_names.index(primary_name)
    primary_sv_tr_full = ensure_binary(np.array(data[cfg["sens_attrs"][primary_idx][0]]))
    primary_sv_te      = ensure_binary(np.array(data[cfg["sens_attrs"][primary_idx][1]]))

    tr_idx, val_idx = build_val_split(y_train, frac=0.15)
    X_tr,  X_val    = X_train[tr_idx], X_train[val_idx]
    y_tr,  y_val    = y_train[tr_idx], y_train[val_idx]
    primary_sv_tr   = primary_sv_tr_full[tr_idx]
    primary_sv_val  = primary_sv_tr_full[val_idx]

    tqdm.write(f"\n{'#'*70}")
    tqdm.write(f"  DATASET: {ds_key.upper()}  |  primary={primary_name}  |  method={method}")
    tqdm.write(f"  n_train={len(y_tr)}, n_val={len(y_val)}, n_test={len(y_test)}")

    set_seed()
    base_enc    = Encoder(X_tr.shape[1], p["hidden_dim"], p["dropout"]).to(DEVICE)
    base_th     = TaskHead(p["hidden_dim"] // 2).to(DEVICE)
    base_opt    = optim.AdamW(
        list(base_enc.parameters()) + list(base_th.parameters()),
        lr=p["lr"], weight_decay=1e-4,
    )
    base_bce    = nn.BCEWithLogitsLoss()
    Xt_b        = torch.tensor(X_tr,   dtype=torch.float32).to(DEVICE)
    Xte_b       = torch.tensor(X_test, dtype=torch.float32).to(DEVICE)
    yt_b        = torch.tensor(y_tr,   dtype=torch.float32).to(DEVICE)
    base_loader = DataLoader(TensorDataset(Xt_b, yt_b), batch_size=p["batch"],
                             shuffle=True, drop_last=True)
    for _ in range(10):
        base_enc.train(); base_th.train()
        for xb, yb in base_loader:
            base_opt.zero_grad(set_to_none=True)
            base_bce(base_th(base_enc(xb)), yb).backward()
            base_opt.step()
    base_enc.eval(); base_th.eval()
    with torch.no_grad():
        base_proba = torch.sigmoid(base_th(base_enc(Xte_b))).cpu().numpy()
    baseline_eo  = compute_eo(y_test, (base_proba > 0.5).astype(int), primary_sv_te)
    baseline_acc = accuracy_score(y_test, (base_proba > 0.5).astype(int))
    tqdm.write(f"  Baseline: EO={baseline_eo:.4f}  acc={baseline_acc:.4f}")
    del base_enc, base_th, base_opt
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    encoder, task_head, final_eo, final_acc = adversarial_training(
        X_tr, y_tr, primary_sv_tr,
        X_val, y_val, primary_sv_val,
        X_test, y_test, primary_sv_te,
        primary_name, p, baseline_eo,
        method=method,
    )

    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    return {
        "baseline_eo": baseline_eo, "baseline_acc": baseline_acc,
        "final_eo":    final_eo,    "final_acc":    final_acc,
    }


def preprocess_german(csv_path, use_cache=True):
    cache = "/tmp/fair_german.pkl"
    if use_cache and os.path.exists(cache): return joblib.load(cache)
    cols = ["status","duration","credit_history","purpose","amount","savings",
            "employment","installment_rate","personal_status_sex","other_debtors",
            "residence","property","age","other_installment_plans","housing",
            "number_credits","job","people_liable","telephone","foreign_worker","credit"]
    df = pd.read_csv(csv_path, sep=" ", names=cols)
    sex_map = {"A91":"male","A92":"male","A93":"male","A94":"female","A95":"female"}
    df["sex"]          = df["personal_status_sex"].map(sex_map)
    df["sex_binary"]   = df["sex"].map({"male":0,"female":1}).fillna(0).astype(int)
    df["age_binary"]   = (df["age"] >= 35).astype(int)
    df["credit_label"] = df["credit"].map({1:1,2:0})
    df = df.drop(columns=["personal_status_sex","sex","foreign_worker","credit"])
    df["log_amount"]        = np.log1p(df["amount"].clip(lower=0))
    df["amount_x_duration"] = df["amount"] * df["duration"]
    num_cols = ["duration","amount","installment_rate","residence","number_credits",
                "people_liable","age","log_amount","amount_x_duration"]
    cat_cols = [c for c in df.columns
                if c not in num_cols + ["credit_label","sex_binary","age_binary"]]
    for c in num_cols: df[c] = clean_numeric(df[c])
    preproc = ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ])
    X  = df.drop(columns=["credit_label"])
    y  = df["credit_label"].values
    sa = df["age_binary"].values
    ss = df["sex_binary"].values
    tr, te = stratified_split(X.values, y, [sa, ss])
    res = {
        "X_train_nn": preproc.fit_transform(X.iloc[tr].reset_index(drop=True)),
        "X_test_nn":  preproc.transform(X.iloc[te].reset_index(drop=True)),
        "y_train": y[tr], "y_test": y[te],
        "sens_age_train": sa[tr], "sens_age_test": sa[te],
        "sens_sex_train": ss[tr], "sens_sex_test": ss[te],
    }
    joblib.dump(res, cache); return res


def preprocess_compas(csv_path, use_cache=True):
    cache = "/tmp/fair_compas.pkl"
    if use_cache and os.path.exists(cache): return joblib.load(cache)
    df = pd.read_csv(csv_path)
    df = df.loc[
        (df["days_b_screening_arrest"].between(-30, 30)) &
        (df["is_recid"] != -1) & (df["c_charge_degree"] != "O") &
        (df["score_text"] != "N/A"),
        ["age","c_charge_degree","race","age_cat","sex","priors_count",
         "days_b_screening_arrest","juv_other_count","juv_misd_count",
         "juv_fel_count","c_charge_desc","is_recid","two_year_recid",
         "c_jail_in","c_jail_out"]
    ].reset_index(drop=True)
    df["race_binary"] = (df["race"] == "African-American").astype(int)
    df["sex_binary"]  = (df["sex"] == "Female").astype(int)
    df["c_jail_in"]   = pd.to_datetime(df["c_jail_in"],  errors="coerce")
    df["c_jail_out"]  = pd.to_datetime(df["c_jail_out"], errors="coerce")
    df["jail_time"]   = (df["c_jail_out"] - df["c_jail_in"]).dt.days.fillna(0).clip(lower=0)
    df = df.drop(columns=["c_jail_in","c_jail_out"])
    df["log_priors"]  = np.log1p(df["priors_count"])
    df["young_adult"] = (df["age"] < 25).astype(int)
    num_cols = ["age","priors_count","days_b_screening_arrest","jail_time",
                "juv_other_count","juv_misd_count","juv_fel_count","log_priors","young_adult"]
    cat_cols = ["c_charge_degree","race","age_cat","sex","c_charge_desc"]
    for c in num_cols: df[c] = clean_numeric(df[c])
    preproc = ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ])
    X  = df.drop(columns=["is_recid","two_year_recid","race_binary","sex_binary"])
    y  = df["two_year_recid"].values
    sr = df["race_binary"].values
    ss = df["sex_binary"].values
    tr, te = stratified_split(X.values, y, [sr, ss])
    res = {
        "X_train_nn": preproc.fit_transform(X.iloc[tr].reset_index(drop=True)),
        "X_test_nn":  preproc.transform(X.iloc[te].reset_index(drop=True)),
        "y_train": y[tr], "y_test": y[te],
        "sens_race_train": sr[tr], "sens_race_test": sr[te],
        "sens_sex_train":  ss[tr], "sens_sex_test":  ss[te],
    }
    joblib.dump(res, cache); return res


def preprocess_adult(csv_path, use_cache=True):
    cache = "/tmp/fair_adult.pkl"
    if use_cache and os.path.exists(cache): return joblib.load(cache)
    df = pd.read_csv(csv_path, skipinitialspace=True)
    df.columns = [c.strip().replace(".", "_") for c in df.columns]
    for c in df.select_dtypes("object").columns:
        df[c] = df[c].str.strip()
    df["income_binary"] = (~df["income"].isin(["<=50K","<=50K."])).astype(int)
    df["sex_binary"]    = (df["sex"] == "Male").astype(int)
    df["race_binary"]   = (df["race"] == "White").astype(int)
    num_cols = ["age","fnlwgt","education_num","capital_gain","capital_loss","hours_per_week"]
    cat_cols = ["workclass","education","marital_status","occupation","relationship","native_country"]
    for c in num_cols: df[c] = clean_numeric(df[c])
    df["log_capital_gain"] = np.log1p(df["capital_gain"].clip(lower=0))
    df["log_capital_loss"] = np.log1p(df["capital_loss"].clip(lower=0))
    df["age_hours"]        = df["age"] * df["hours_per_week"]
    num_cols = num_cols + ["log_capital_gain","log_capital_loss","age_hours"]
    preproc = ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ])
    drop_cols = {"income","income_binary","sex_binary","race_binary","race","sex"}
    X  = df.drop(columns=[c for c in drop_cols if c in df.columns])
    y  = df["income_binary"].values
    ss = df["sex_binary"].values
    sr = df["race_binary"].values
    tr, te = stratified_split(X.values, y, [ss, sr])
    res = {
        "X_train_nn": preproc.fit_transform(X.iloc[tr].reset_index(drop=True)),
        "X_test_nn":  preproc.transform(X.iloc[te].reset_index(drop=True)),
        "y_train": y[tr], "y_test": y[te],
        "sens_sex_train":  ss[tr], "sens_sex_test":  ss[te],
        "sens_race_train": sr[tr], "sens_race_test": sr[te],
    }
    joblib.dump(res, cache); return res


def preprocess_communities(csv_path, use_cache=True):
    cache = "/tmp/fair_communities.pkl"
    if use_cache and os.path.exists(cache): return joblib.load(cache)
    CANONICAL_NAMES = [
        "state","county","community","communityname","fold",
        "population","householdsize","racepctblack","racePctWhite",
        "racePctAsian","racePctHisp","agePct12t21","agePct12t29",
        "agePct16t24","agePct65up","numbUrban","pctUrban","medIncome",
        "pctWWage","pctWFarmSelf","pctWInvInc","pctWSocSec","pctWPubAsst",
        "pctWRetire","medFamInc","perCapInc","whitePerCap","blackPerCap",
        "indianPerCap","AsianPerCap","OtherPerCap","HispPerCap",
        "NumUnderPov","PctPopUnderPov","PctLess9thGrade","PctNotHSGrad",
        "PctBSorMore","PctUnemployed","PctEmploy","PctEmplManu",
        "PctEmplProfServ","PctOccupManu","PctOccupMgmtProf","MalePctDivorce",
        "MalePctNevMarr","FemalePctDiv","TotalPctDiv","PersPerFam",
        "PctFam2Par","PctKids2Par","PctYoungKids2Par","PctTeen2Par",
        "PctWorkMomYoungKids","PctWorkMom","NumIlleg","PctIlleg",
        "NumImmig","PctImmigRecent","PctImmigRec5","PctImmigRec8",
        "PctImmigRec10","PctRecentImmig","PctRecImmig5","PctRecImmig8",
        "PctRecImmig10","PctSpeakEnglOnly","PctNotSpeakEnglWell",
        "PctLargHouseFam","PctLargHouseOccup","PersPerOccupHous",
        "PersPerOwnOccHous","PersPerRentOccHous","PctPersOwnOccup",
        "PctPersDenseHous","PctHousLess3BR","MedNumBR","HousVacant",
        "PctHousOccup","PctHousOwnOcc","PctVacantBoarded","PctVacMore6Mos",
        "MedYrHousBuilt","PctHousNoPhone","PctWOFullPlumb","OwnOccLowQuart",
        "OwnOccMedVal","OwnOccHiQuart","OwnOccQrange","RentLowQ",
        "RentMedian","RentHighQ","RentQrange","MedRent","MedRentPctHousInc",
        "MedOwnCostPctInc","MedOwnCostPctIncNoMtg","NumInShelters",
        "NumStreet","PctForeignBorn","PctBornSameState","PctSameHouse85",
        "PctSameCity85","PctSameState85","LemasSwFTPerPop",
        "LemasSwFTFieldPerPop","LemasTotalReq","LemasTotReqPerPop",
        "PolicReqPerOffic","PolicPerPop","RacialMatchCommPol","PctPolicWhite",
        "PctPolicBlack","PctPolicHisp","PctPolicAsian","PctPolicMinor",
        "OfficAssgnDrugUnits","NumKindsDrugsSeiz","PolicAveOTWorked",
        "LandArea","PopDens","PctUsePubTrans","PolicCars","PolicOperBudg",
        "LemasPctPolicOnPatr","LemasGangUnitDeploy","LemasPctOfficDrugUn",
        "PolicBudgPerPop","ViolentCrimesPerPop",
    ]
    RACE_COLS = {
        "racepctblack","racePctWhite","racePctAsian","racePctHisp",
        "whitePerCap","blackPerCap","indianPerCap","AsianPerCap",
        "OtherPerCap","HispPerCap","PctPolicWhite","PctPolicBlack",
        "PctPolicHisp","PctPolicAsian","PctPolicMinor","RacialMatchCommPol",
    }
    raw = pd.read_csv(csv_path, header=None, dtype=str)
    if raw.shape[1] == 2:
        rows = []
        for val in raw.iloc[:, 1]:
            if pd.isna(val): continue
            parts = str(val).split(",")
            if len(parts) == len(CANONICAL_NAMES):
                rows.append(parts)
        if not rows: raise ValueError("[communities] No valid rows found in col 1")
        df = pd.DataFrame(rows, columns=CANONICAL_NAMES)
    elif raw.shape[1] == len(CANONICAL_NAMES):
        df = raw.copy(); df.columns = CANONICAL_NAMES
    else:
        raise ValueError(f"[communities] Unexpected shape {raw.shape}.")
    tqdm.write(f"  [communities] loaded {len(df)} rows")
    target_col = "ViolentCrimesPerPop"
    race_col   = "racepctblack"
    drop_ids   = ["state","county","community","communityname","fold"]
    df = df.drop(columns=[c for c in drop_ids if c in df.columns])
    df = df.replace("?", np.nan)
    df[target_col] = pd.to_numeric(df[target_col], errors="coerce")
    df[race_col]   = pd.to_numeric(df[race_col],   errors="coerce")
    df = df.dropna(subset=[target_col, race_col]).reset_index(drop=True)
    crime_thresh   = df[target_col].quantile(0.60)
    race_thresh    = df[race_col].quantile(0.60)
    df["crime_label"] = (df[target_col] > crime_thresh).astype(int)
    df["race_binary"] = (df[race_col]   > race_thresh).astype(int)
    exclude_feats = {target_col, "crime_label", "race_binary"} | RACE_COLS
    feat_cols     = [c for c in df.columns if c not in exclude_feats]
    for c in feat_cols: df[c] = pd.to_numeric(df[c], errors="coerce")
    df[feat_cols] = df[feat_cols].fillna(df[feat_cols].median())
    keep    = [c for c in feat_cols if df[c].std() > 1e-6]
    preproc = ColumnTransformer([("num", StandardScaler(), keep)])
    X  = df[keep]
    y  = df["crime_label"].values
    sr = df["race_binary"].values
    tr, te = stratified_split(X.values, y, [sr])
    res = {
        "X_train_nn": preproc.fit_transform(X.iloc[tr].reset_index(drop=True)),
        "X_test_nn":  preproc.transform(X.iloc[te].reset_index(drop=True)),
        "y_train": y[tr], "y_test": y[te],
        "sens_race_train": sr[tr], "sens_race_test": sr[te],
    }
    joblib.dump(res, cache); return res


def preprocess_heart(csv_path, use_cache=True):
    cache = "/tmp/fair_heart.pkl"
    if use_cache and os.path.exists(cache): return joblib.load(cache)
    df = pd.read_csv(csv_path)
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
    target_col = None
    for c in ["num","death_event","deathevent","target","outcome","event"]:
        if c in df.columns: target_col = c; break
    if target_col is None: target_col = df.columns[-1]
    df["label"] = (pd.to_numeric(df[target_col], errors="coerce").fillna(0) > 0).astype(int)
    sex_col = next((c for c in ["sex","gender"] if c in df.columns), None)
    if sex_col is None: raise ValueError("[heart] No sex/gender column found")
    sex_raw = df[sex_col]
    if sex_raw.dtype == object or str(sex_raw.dtype) == "string":
        df["sex_binary"] = sex_raw.str.strip().str.lower().isin(["male","1"]).astype(int)
    else:
        df["sex_binary"] = pd.to_numeric(sex_raw, errors="coerce").fillna(0).astype(int)
    age_col = next((c for c in df.columns if "age" in c), None)
    if age_col is None: raise ValueError("[heart] No age column found")
    df[age_col]      = pd.to_numeric(df[age_col], errors="coerce")
    df["age_binary"] = (df[age_col] > df[age_col].median()).astype(int)
    exclude        = {target_col,"label","sex_binary","age_binary",sex_col,"id","dataset"}
    bool_like_cats = {"fbs","exang"}
    string_cats    = {"cp","restecg","slope","thal"}
    num_cols = []; cat_cols = []
    for c in df.columns:
        if c in exclude: continue
        if c in bool_like_cats or c in string_cats or df[c].dtype == object:
            cat_cols.append(c)
        else:
            num_cols.append(c)
    for c in num_cols:
        s = pd.to_numeric(df[c], errors="coerce")
        df[c] = s.fillna(s.median() if s.notna().any() else 0)
    for c in cat_cols:
        df[c] = df[c].astype(str).replace("nan","missing").replace("NaN","missing")
    num_cols = [c for c in num_cols if df[c].std() > 1e-6]
    tqdm.write(f"  [heart] num_cols={num_cols}  cat_cols={cat_cols}")
    preproc = ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ])
    feat_cols = num_cols + cat_cols
    X  = df[feat_cols]
    y  = df["label"].values
    ss = df["sex_binary"].values
    sa = df["age_binary"].values
    tr, te = stratified_split(X.values, y, [ss, sa])
    res = {
        "X_train_nn": preproc.fit_transform(X.iloc[tr].reset_index(drop=True)),
        "X_test_nn":  preproc.transform(X.iloc[te].reset_index(drop=True)),
        "y_train": y[tr], "y_test": y[te],
        "sens_sex_train": ss[tr], "sens_sex_test": ss[te],
        "sens_age_train": sa[tr], "sens_age_test": sa[te],
    }
    joblib.dump(res, cache); return res


def main():
    for f in ["/tmp/fair_german.pkl","/tmp/fair_compas.pkl","/tmp/fair_adult.pkl",
              "/tmp/fair_communities.pkl","/tmp/fair_heart.pkl"]:
        if os.path.exists(f): os.remove(f)

    DATA_PATHS = {
        "adult":       "/kaggle/input/datasets/lateglue/all-datasets/adult.csv",
        "communities": "/kaggle/input/datasets/murakri1994/uci-communities-and-crimes-dataset/CollatedData.csv",
        "heart":       "/kaggle/input/datasets/redwankarimsony/heart-disease-data/heart_disease_uci.csv",
        "german":      "/kaggle/input/datasets/lateglue/all-datasets/german.data",
        "compas":      "/kaggle/input/datasets/lateglue/all-datasets/compas-scores-two-years.csv",
    }
    PREPROCESS_FNS = {
        "adult":       preprocess_adult,
        "communities": preprocess_communities,
        "heart":       preprocess_heart,
        "german":      preprocess_german,
        "compas":      preprocess_compas,
    }

    METHODS = ["deterministic", "bayesian"]

    print(f"\nAblation: Deterministic vs Bayesian Adversarial Debiasing")
    print(f"TARGET_EO={TARGET_EO}  |  MIN_ACC_FLOOR={MIN_ACC_FLOOR}  |  Device={DEVICE}")
    print(f"N_ADV_SAMPLES={N_ADV_SAMPLES}  N_ENC_SAMPLES={N_ENC_SAMPLES}  N_TEST_SAMPLES={N_TEST_SAMPLES}")
    print("="*70)

    all_results = {m: {} for m in METHODS}

    for method in METHODS:
        print(f"\n{'*'*70}")
        print(f"  METHOD: {method.upper()}")
        print(f"{'*'*70}")
        for ds_key in DATASET_CONFIG:
            for f in ["/tmp/fair_german.pkl","/tmp/fair_compas.pkl","/tmp/fair_adult.pkl",
                      "/tmp/fair_communities.pkl","/tmp/fair_heart.pkl"]:
                if os.path.exists(f): os.remove(f)
            data = PREPROCESS_FNS[ds_key](DATA_PATHS[ds_key])
            all_results[method][ds_key] = run_pipeline(data, ds_key, method=method)

    print(f"\n{'='*85}")
    print(f"ABLATION SUMMARY  (PASS = EO < {TARGET_EO})")
    print(f"{'─'*85}")
    print(f"  {'Dataset':<14} {'Primary':<8} "
          f"{'Det EO':>8} {'Det Acc':>8} {'Det':>5}  "
          f"{'Bay EO':>8} {'Bay Acc':>8} {'Bay':>5}  {'ΔΔEO':>8}")
    print(f"  {'─'*83}")
    for ds_key in DATASET_CONFIG:
        pn  = DATASET_CONFIG[ds_key]["primary_sens"]
        det = all_results["deterministic"][ds_key]
        bay = all_results["bayesian"][ds_key]
        det_status  = "PASS" if det["final_eo"] < TARGET_EO else "FAIL"
        bay_status  = "PASS" if bay["final_eo"] < TARGET_EO else "FAIL"
        delta_delta = bay["final_eo"] - det["final_eo"]
        print(f"  {ds_key.upper():<14} {pn:<8} "
              f"{det['final_eo']:>8.4f} {det['final_acc']:>8.3f} {det_status:>5}  "
              f"{bay['final_eo']:>8.4f} {bay['final_acc']:>8.3f} {bay_status:>5}  "
              f"{delta_delta:>+8.4f}")
    print("="*85)


main()


Ablation: Deterministic vs Bayesian Adversarial Debiasing
TARGET_EO=0.05  |  MIN_ACC_FLOOR=0.55  |  Device=cuda
N_ADV_SAMPLES=10  N_ENC_SAMPLES=16  N_TEST_SAMPLES=40

**********************************************************************
  METHOD: DETERMINISTIC
**********************************************************************

######################################################################
  DATASET: ADULT  |  primary=sex  |  method=deterministic
  n_train=22141, n_val=3907, n_test=6513
  Baseline: EO=0.0839  acc=0.8549

  [DETERMINISTIC] Adv Training  |  sens=sex  |  baseline_EO=0.0839
  Pre-training adversary (5 epochs)...


  [det]:   0%|          | 0/1000 [00:00<?, ?it/s]


  Early stop ep=635
  Baseline→Final: EO 0.0839 → 0.0504  acc=0.8497  FAIL
  [communities] loaded 1994 rows

######################################################################
  DATASET: COMMUNITIES  |  primary=race  |  method=deterministic
  n_train=1356, n_val=239, n_test=399
  Baseline: EO=0.3325  acc=0.8496

  [DETERMINISTIC] Adv Training  |  sens=race  |  baseline_EO=0.3325
  Pre-training adversary (30 epochs)...


  [det]:   0%|          | 0/3000 [00:00<?, ?it/s]


  Early stop ep=725
  Baseline→Final: EO 0.3325 → 0.1070  acc=0.7945  FAIL
  [heart] num_cols=['age', 'trestbps', 'chol', 'thalch', 'oldpeak', 'ca']  cat_cols=['cp', 'fbs', 'restecg', 'exang', 'slope', 'thal']

######################################################################
  DATASET: HEART  |  primary=sex  |  method=deterministic
  n_train=626, n_val=110, n_test=184
  Baseline: EO=0.1295  acc=0.7880

  [DETERMINISTIC] Adv Training  |  sens=sex  |  baseline_EO=0.1295
  Pre-training adversary (20 epochs)...


...

  Baseline→Final: EO 0.1295 → 0.0522  acc=0.8098  FAIL

######################################################################
  DATASET: GERMAN  |  primary=age  |  method=bayesian
  n_train=680, n_val=120, n_test=200
  Baseline: EO=0.1077  acc=0.6850

  [BAYESIAN] Adv Training  |  sens=age  |  baseline_EO=0.1077
  Pre-training adversary (20 epochs)...


  [bay]:   0%|          | 0/3000 [00:00<?, ?it/s]

  Baseline→Final: EO 0.1077 → 0.1029  acc=0.6850  FAIL

######################################################################
  DATASET: COMPAS  |  primary=race  |  method=bayesian
  n_train=4197, n_val=740, n_test=1235
  Baseline: EO=0.2222  acc=0.6899

  [BAYESIAN] Adv Training  |  sens=race  |  baseline_EO=0.2222
  Pre-training adversary (20 epochs)...


  [bay]:   0%|          | 0/2000 [00:00<?, ?it/s]

  Baseline→Final: EO 0.2222 → 0.0482  acc=0.6672  PASS

ABLATION SUMMARY  (PASS = EO < 0.05)
─────────────────────────────────────────────────────────────────────────────────────
  Dataset        Primary    Det EO  Det Acc   Det    Bay EO  Bay Acc   Bay      ΔΔEO
  ───────────────────────────────────────────────────────────────────────────────────
  ADULT          sex        0.0504    0.850  FAIL    0.0352    0.776  PASS   -0.0151
  COMMUNITIES    race       0.1070    0.794  FAIL    0.3229    0.812  FAIL   +0.2159
  HEART          sex        0.1483    0.766  FAIL    0.0522    0.810  FAIL   -0.0962
  GERMAN         age        0.0084    0.690  PASS    0.1029    0.685  FAIL   +0.0945
  COMPAS         race       0.0281    0.651  PASS    0.0482    0.667  PASS   +0.0202
